## Empowering Topic Representation with LLM

In [1]:
import openai
import pandas as pd

from bertopic.representation import OpenAI
from bertopic import BERTopic

from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_csv('hotel_reviews_with_transl.csv', sep = '\t')
df.sample(3)

,id,hotel,review,lang,reviews_transl,reviews_len
7319,7692,Travelodge,Enjoyed our budget stay Stayed here for two ni...,en,Enjoyed our budget stay Stayed here for two ni...,943
6621,6975,Millemiun,Fantastic Hotel great location Very impressed ...,en,Fantastic Hotel great location Very impressed ...,635
2142,2269,Radisson,"Excellent location This hotel is great, the ro...",en,"Excellent location This hotel is great, the ro...",199


In [3]:
docs = list(df.reviews_transl)
docs[:3]

["We stayed three nights over Thanksgiving weekend. While it's a ways out of downtown London, the little town it's in is great--lots of delicious restaurants, inexpensive shopping, close to a fairly major highway. Our two rooms were spare but comfortable and very clean. Bathrooms were fine. Mattresses were sad, saggy and lumpy, but at least not rock hard and the sheets fit and stayed on well. The comforter was just the perfect weight for the weather--we all slept well. There's an in-house restaurant with room service, which we never had a chance to use. The a la carte prices were reasonable, except the buffet breakfast, which was high. It's difficult to find the parking lots, but once you figure out the one way streets and round-abouts, there is safe, adequate parking. We'd go back. For only nine pounds a night for two people, it's a great bargain!",
 "very uncomfortable stay We were staying at his hotel only because my father couldnt make it and rather than cancel the rooms we took hi

In [4]:
client = openai.OpenAI(
    # Ollama の OpenAI 互換 API は /v1 パス配下で提供される
    # macOS の Docker は Apple Silicon GPU (Metal) にアクセスできないため、
    # Ollama はホスト側でネイティブ実行し、host.docker.internal 経由で接続する
    base_url = 'http://host.docker.internal:11434/v1',
    api_key = 'ollama',
)

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
# sentence_model = SentenceTransformer("intfloat/multilingual-e5-large")

# BERTopic の OpenAI RepresentationModel を qwen3 で使う際の注意点:
#
# 1. stop トークンの上書き
#    BERTopic はデフォルトで stop="\n" を設定するため、LLM の出力が最初の改行で
#    打ち切られてしまう。特に qwen3 は thinking mode がデフォルト有効で、
#    レスポンスが "<think>\n" で始まるため即座に停止し空ラベルになる。
#    generator_kwargs で truthy な値 (例: "<|im_end|>") を渡して上書きする。
#    ※ stop=None は falsy なため BERTopic 内部で "\n" に再設定されてしまう。
#
# 2. qwen3 の thinking mode 無効化
#    プロンプト末尾に /no_think を付与して thinking mode を無効化する。
#    これにより <think>...</think> タグが出力されず、ラベルが正しくパースされる。

prompt = """You will extract a short topic label from given documents and keywords.
Here are two examples of topics you created before:

# Example 1
Sample texts from this topic:
- Traditional diets in most cultures were primarily plant-based with a little meat on top, but with the rise of industrial style meat production and factory farming, meat has become a staple food.
- Meat, but especially beef, is the worst food in terms of emissions.
- Eating meat doesn't make you a bad person, not eating meat doesn't make you a good one.

Keywords: meat beef eat eating emissions steak food health processed chicken
topic: Environmental impacts of eating meat

# Example 2
Sample texts from this topic:
- I have ordered the product weeks ago but it still has not arrived!
- The website mentions that it only takes a couple of days to deliver but I still have not received mine.
- I got a message stating that I received the monitor but that is not true!
- It took a month longer to deliver than was advised...

Keywords: deliver weeks product shipping long delivery received arrived arrive week
topic: Shipping and delivery issues

# Your task
Sample texts from this topic:
[DOCUMENTS]

Keywords: [KEYWORDS]

Based on the information above, extract a short topic label (three words at most) in the following format:
topic: <topic_label>
/no_think"""

representation_model = OpenAI(
    client,
    model='qwen3:1.7b',
    prompt=prompt,
    nr_docs=2,
    doc_length=200,
    tokenizer='whitespace',
    generator_kwargs={"stop": "<|im_end|>"},
)

topic_model = BERTopic(
    embedding_model=sentence_model,
    representation_model=representation_model,
    verbose=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
topics, ini_probs = topic_model.fit_transform(docs)

2026-04-13 07:54:58,646 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/383 [00:00<?, ?it/s]

2026-04-13 07:58:07,644 - BERTopic - Embedding - Completed ✓
2026-04-13 07:58:07,645 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-13 07:58:16,568 - BERTopic - Dimensionality - Completed ✓
2026-04-13 07:58:16,569 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 07:58:16,828 - BERTopic - Cluster - Completed ✓
2026-04-13 07:58:16,832 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 106/106 [38:22<00:00, 21.72s/it]
2026-04-13 08:36:40,723 - BERTopic - Representation - Completed ✓


In [6]:
topic_model.get_topic_info().head(10).set_index('Topic')[['Count', 'Name', 'Representation']]

,Count,Name,Representation
Topic,,,
-1,6738,-1_Hotel room issues,[Hotel room issues]
0,266,0_Hotel noise and room,[Hotel noise and room]
1,246,1_Hotel Experience,[Hotel Experience]
2,238,2_Hotel Location and Services,[Hotel Location and Services]
3,226,3_Hotel experience,[Hotel experience]
4,194,4_Hotel reviews,[Hotel reviews]
5,184,5_Hotel Location and Amenities,[Hotel Location and Amenities]
6,153,6_Hotel value and cleanliness,[Hotel value and cleanliness]
7,136,7_Hotel experiences,[Hotel experiences]
